# Dataset Curation 

some of our dft calculations could have gone wrong. The way to filter this is through the E-V curves for each sample. 

# 
- input : non curated bs
- output: curated bs regarding ev curves

TODO: bopio and bopcal needed for featurizer ! separate package for bopfox ?

## check ev-curves for goodness

In [1]:
import sys
#sys.path.insert(0,'/home/storage/fortimtb/DatasetsML/Tools/')
sys.path.insert(0,'/home/storage/fortimtb/CuadernoTrabajo/bopfoxfeaturizer/')
dataset = 'Fe-Mo/'
from Tools.DatasetTools.EVCurvesTools import *
from BopFoxFeaturizer.Featurizer import Featurizer
import pickle


import os

In [2]:
PBS = pd.read_pickle(os.path.join(dataset, 'ParsedBriefsummary.pkl'))

## Investigate ev-curves

In [3]:
evcurvesloc = os.path.join(dataset,'evcurves.json' )
if not os.path.exists(evcurvesloc):
    EV = Evcurves(atoms=['Fe','Mo'], dataset = dataset)#, search_str='**/volume_relaxed/**/volume-energy.dat')
    EV.load_evcurves(PBS.index, deltaks = PBS['deltak'], encuts = PBS['encut'])
    EV.evcurves.to_json(evcurvesloc)
EVcurves = pd.read_json(evcurvesloc, typ='series') 
goodness, fiteos, r2  = get_goodness(EVcurves)         
#plot_curves (goodness, EVcurves, 'multipage_fitted_curves.pdf')
#plot_curves (invert_goodness(goodness), EVcurves, 'multipage_bad_curves.pdf')

EmptyDataError: No columns to parse from file

In [4]:
goodness

NameError: name 'goodness' is not defined

In [5]:
istheregoodness = 'goodness' in globals()
if istheregoodness:
    indexofgoodsamples = pd.Index([theindex for theindex, thecurve in goodness.items() if True in thecurve.values()])
else:
    indexofgoodsamples = PBS.index
GoodBS = PBS.loc[indexofgoodsamples]
BadBS = PBS.loc[PBS.index.difference(indexofgoodsamples)]

## Remove extra magnetic sampling

First feature to remove from this dataset is the list of samples used form ferrimagnetic phase sampling. This subset is not in the current interest and might bring problems so we remove from datastet.

In [6]:
GoodBS = GoodBS.loc[~GoodBS.index.str.contains('\..*[UD]+$') ]

In [7]:
GoodBS.info(verbose=False)

<class 'pandas.core.frame.DataFrame'>
Index: 292 entries, Fe_pv8Mo_sv22.sigma-BBABB.FM to Fe_pv10Mo_sv20.sigma-ABBAB.NM
Columns: 22 entries,  to EF
dtypes: float64(6), int64(1), object(15)
memory usage: 52.5+ KB


## Obtain some info from indexes

In [8]:
Features = Featurizer(GoodBS)

## translate structures to their bases

# TODo this sould be in tools, as a phase cleaner

In [9]:
from BopFoxFeaturizer.struct_db import struct_db
#struct_db = SourceFileLoader('struct_db','BopFoxFeaturizer/struct_db.py').load_module().struct_db
strucdic = struct_db().strucstrings

Target_Class = pd.Series(
    GoodBS.index.str.split('.').map(lambda l: l[1]).map(lambda s: s.split('-')[0]),
    index=GoodBS.index
)
Target_Class[Target_Class.map(lambda s: s in strucdic['list.hcp'])]='hcp'
Target_Class[Target_Class.map(lambda s: s in strucdic['list.fcc'])]='fcc'
Target_Class[Target_Class.map(lambda s: s in strucdic['list.bcc'])]='bcc'
Target_Class[Features.Struc == 'hcp'] = 'hcp'
Target_Class[Features.Struc == 'bcc'] = 'bcc'
Target_Class[Features.Struc == 'fcc'] = 'fcc'
Target_Class[Features.Struc.str.contains('SQS-fcc')] = 'fcc'
Target_Class[Features.Struc.str.contains('SQS-L12')] = 'fcc'
Target_Class[Features.Struc.str.contains('sigma_')] = 'sigma'

Target_Class[    
    Target_Class.str.contains('Al42W') |\
    Target_Class.str.contains('Al9Co2') |\
    Target_Class.str.contains('Al5W') |\
    Target_Class.str.contains('Al12W') |\
    Target_Class.str.contains('Al4W') |\
    Target_Class.str.contains('Al5Co2')
] = 'others'

In [10]:
Target_Class

index
Fe_pv8Mo_sv22.sigma-BBABB.FM     sigma
Fe_pv10Mo_sv20.sigma-ABBAB.FM    sigma
Fe_pv4Mo_sv20.C36-ABBBB.FM         C36
Fe_pv3Mo_sv10.mu-ABBBA.FM           mu
Fe_pv5Mo_sv24.chi-AABB.FM          chi
                                 ...  
Fe_pv3Mo_sv10.mu-ABBBA.NM           mu
Fe_pv8Mo_sv22.sigma-BBABB.NM     sigma
Fe_pv1Mo_sv3.L12-AB3.FM            fcc
Fe_pv8Mo_sv22.sigma-BBBBA.FM     sigma
Fe_pv10Mo_sv20.sigma-ABBAB.NM    sigma
Name: index, Length: 292, dtype: object

In [11]:
GoodBS['Phase'] = Target_Class

In [12]:
GoodBS.describe()

,E0,nelem,B0,V0,num_atoms,Fe_pv,Mo_sv
count,292.000000,292.000000,292.000000,292.000000,292.000000,292.000000,292.000000
mean,-9.340212,1.859589,200.212323,13.678430,21.147260,0.495522,0.504478
std,0.718179,0.348009,420.142274,1.632573,11.112154,0.280342,0.280342
min,-10.934283,1.000000,-6867.530562,8.318826,1.000000,0.000000,0.000000
25%,-9.910610,2.000000,207.453577,12.489970,13.000000,0.266667,0.307692
50%,-9.327641,2.000000,237.309049,13.687829,24.000000,0.500000,0.500000
75%,-8.856989,2.000000,257.615980,14.933957,30.000000,0.692308,0.733333
max,-7.780040,2.000000,420.901429,17.320677,56.000000,1.000000,1.000000


# Save for later use 

In [13]:
curatedbs = os.path.join(dataset,'CuratedParsedBriefSummary.pkl')
GoodBS.to_pickle(curatedbs)

# some E-V curves, good and bad

In [14]:
sample_bad = EVcurves[BadBS.index].dropna().sample(n=5)

NameError: name 'EVcurves' is not defined

In [15]:
sample_bad

NameError: name 'sample_bad' is not defined

In [16]:
goodness[sample_bad.index]

NameError: name 'goodness' is not defined

In [17]:
sample_bad_r2 = r2[sample_bad.index]

NameError: name 'r2' is not defined

In [18]:
sample_bad_fit = fiteos[sample_bad.index]

NameError: name 'fiteos' is not defined

In [19]:
figurecollection, axcollection  = plot_curves(sample_bad, sample_bad_fit, sample_bad_r2)

NameError: name 'sample_bad' is not defined

In [20]:
sample_good = EVcurves[GoodBS.index].dropna().sample(n=5)

NameError: name 'EVcurves' is not defined

In [21]:
sample_good

NameError: name 'sample_good' is not defined

In [22]:
goodness[sample_good.index]

NameError: name 'goodness' is not defined

In [23]:
sample_good_r2 = r2[sample_good.index]

NameError: name 'r2' is not defined

In [24]:
sample_good_fit = fiteos[sample_good.index]

NameError: name 'fiteos' is not defined

In [25]:
figurecollection, axcollection  = plot_curves(sample_good, sample_good_fit, sample_good_r2)

NameError: name 'sample_good' is not defined